# HYZ 2026 Competition Colab

> **Notebook release:** `2026-07-16 15:19:55 Europe/Istanbul (+03:00)` — cache-bust `HYZ-COLAB-20260716-151955`

Bu notebook private ana repoyu ve dört submodule'u hazırlar, Task 1 object detector modelini doğrulanmış kaynaktan indirir, `cls.pt` landing classifier modelini bilgisayarından manuel yükler ve gerçek Task 1 + Task 2 + Task 3 yarışma runner'ını başlatır.

> **Önemli:** Bu notebook operasyonel yarışma akışını hazırlar; algoritmik skor garantisi vermez. Task 3 canlı production backend'i DINOv2 semantic acquisition + optical flow tracking kullanır.


## 1. Colab Secrets

Colab sol menüde **Secrets** bölümüne aşağıdaki değerleri ekle ve notebook erişimini aç:

- `GITHUB_TOKEN`
- `TEAM_NAME`
- `PASSWORD`
- `EVALUATION_SERVER_URL`
- `SESSION_NAME` — sunucunun yarışma için verdiği oturum adı

Secret değerleri ekrana basılmaz. `GITHUB_TOKEN` yalnız private repo/submodule indirmek için geçici `GIT_ASKPASS` üzerinden kullanılır.


In [ ]:
import os
import stat
import subprocess
from pathlib import Path
from google.colab import userdata

NOTEBOOK_RELEASE_TIMESTAMP = "2026-07-16 15:19:55 Europe/Istanbul (+03:00)"
NOTEBOOK_CACHE_BUST = "HYZ-COLAB-20260716-151955"
print(f"NOTEBOOK RELEASE: {NOTEBOOK_RELEASE_TIMESTAMP} | {NOTEBOOK_CACHE_BUST}")

REPOSITORY_URL = "https://github.com/RaidynTeam/HYZ_2026.git"
REPOSITORY_BRANCH = "main"
REPOSITORY_DIR = Path("/content/HYZ_2026")
VENV_PYTHON = REPOSITORY_DIR / ".venv/bin/python"
REQUIRED_PARENT_ANCESTOR = "0f16ba3d0a6141796b5028f9c449cae415096ea9"
REQUIRED_TASK_COMMITS = {
    "modules/task1_detection": "32000c44c5bbc5a87ec3ce00d148a99bc498e6d7",
    "modules/task2_trajectory": "3060720bafbf4f3593e8a63a5ebf638dc400005a",
    "modules/task3_reference": "a2650240db1009e0ae2a0dd778a163b9799e2e90",
}

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secrets içinde GITHUB_TOKEN eksik")

os.chdir("/content")
askpass = Path("/content/.hyz_git_askpass.sh")
askpass.write_text(
    '#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n',
    encoding="utf-8",
)
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
git_environment = os.environ.copy()
git_environment.update({
    "GITHUB_TOKEN": str(github_token),
    "GIT_ASKPASS": str(askpass),
    "GIT_TERMINAL_PROMPT": "0",
})

def git(args, *, cwd=REPOSITORY_DIR):
    subprocess.run(["git", *args], cwd=cwd, env=git_environment, check=True)

try:
    if (REPOSITORY_DIR / ".git").is_dir():
        print("Mevcut Colab runtime korunuyor; _logs snapshot ve indirilen kareler silinmeyecek.")
        git(["submodule", "update", "--init", "--recursive"])
        git(["pull", "--ff-only", "origin", REPOSITORY_BRANCH])
    else:
        subprocess.run(
            [
                "git", "clone", "--recurse-submodules",
                "--branch", REPOSITORY_BRANCH, "--single-branch",
                REPOSITORY_URL, str(REPOSITORY_DIR),
            ],
            env=git_environment,
            check=True,
        )
    git(["submodule", "update", "--init", "--recursive"])
    for relative_path, required_commit in REQUIRED_TASK_COMMITS.items():
        module_dir = REPOSITORY_DIR / relative_path
        git(["fetch", "origin"], cwd=module_dir)
        git(["checkout", "--detach", required_commit], cwd=module_dir)
        actual = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=module_dir, text=True
        ).strip()
        if actual != required_commit:
            raise RuntimeError(f"{relative_path} source pin mismatch: {actual}")
    git(
        ["submodule", "update", "--init", "--recursive"],
        cwd=REPOSITORY_DIR / "modules/task3_reference",
    )
finally:
    askpass.unlink(missing_ok=True)
    git_environment.pop("GITHUB_TOKEN", None)
    del github_token

os.chdir(REPOSITORY_DIR)
task2_config_path = REPOSITORY_DIR / "modules/task2_trajectory/config/team_solution.yaml"
task2_config_text = task2_config_path.read_text(encoding="utf-8")
development_marker = "runtime:\n  mode: development"
production_marker = "runtime:\n  mode: production"
if development_marker in task2_config_text:
    task2_config_text = task2_config_text.replace(
        development_marker, production_marker, 1
    )
    task2_config_path.write_text(task2_config_text, encoding="utf-8")
elif production_marker not in task2_config_text:
    raise RuntimeError("Task 2 runtime.mode güvenli biçimde production yapılamadı")
print("Task 2 Colab çalışma config'i: runtime.mode=production")
task3_config_path = REPOSITORY_DIR / "config/task3.yaml"
task3_config_text = task3_config_path.read_text(encoding="utf-8")
task3_config_text = task3_config_text.replace("backend: opencv", "backend: dinov2", 1)
task3_config_text = task3_config_text.replace(
    "weights_path: null",
    "weights_path: ../models/dinov2_vits14_pretrain.pth",
    1,
)
if "backend: dinov2" not in task3_config_text:
    raise RuntimeError("Task 3 backend dinov2 yapılamadı")
task3_config_path.write_text(task3_config_text, encoding="utf-8")
print("Task 3 Colab çalışma config'i: backend=dinov2")
parent_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR, text=True
).strip()
ancestor_check = subprocess.run(
    ["git", "merge-base", "--is-ancestor", REQUIRED_PARENT_ANCESTOR, parent_commit],
    cwd=REPOSITORY_DIR,
)
if ancestor_check.returncode != 0:
    raise RuntimeError(
        f"Ana repo eski: {parent_commit[:8]}; gerekli taban: {REQUIRED_PARENT_ANCESTOR[:8]}"
    )
print(f"Ana repo hazır: {parent_commit[:8]}")
for path, commit in REQUIRED_TASK_COMMITS.items():
    print(f"Kaynak pini: {path} = {commit[:8]}")


## 2. Task 1 ve DINOv2 modelleri

Object detection ve landing classification modelleri aşağıdaki **iki ayrı upload hücresinden** yüklenir. Yüklediğin dosyaların adları önemli değildir; her hücre SHA-256 doğrulamasından sonra modeli yarışma kodunun beklediği sabit ada yeniden yazar. Official DINOv2 ViT-S/14 ağırlığı otomatik indirilir.

Beklenen sınıflar:

- Object detection: `0=vehicle, 1=person, 2=uap, 3=uai`
- Landing classification: `0=landable, 1=not_landable`


In [ ]:
import hashlib
from urllib.request import Request, urlopen
from google.colab import files

TASK1_MODEL_DIR = REPOSITORY_DIR / "modules/task1_detection/hyz_task1/models"
DETECTOR_PATH = TASK1_MODEL_DIR / "yolo26n-1.1.0.pt"
LANDING_CLASSIFIER_PATH = TASK1_MODEL_DIR / "landing_cls.pt"
DINOV2_MODEL_PATH = REPOSITORY_DIR / "models/dinov2_vits14_pretrain.pth"
DETECTOR_SHA256 = "dd79f138c1796f25590a903b1870a37f76efefa9c3b41865a890440dc2d3ec86"
LANDING_CLASSIFIER_SHA256 = "d66c2e20b8fb38ec76aa870b2b6946977a7540543bc4310aef8d3bd078917c5e"
DINOV2_URL = "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth"
DINOV2_SHA256 = "b938bf1bc15cd2ec0feacfe3a1bb553fe8ea9ca46a7e1d8d00217f29aef60cd9"
DINOV2_SIZE_BYTES = 88283115

def sha256_path(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def upload_verified_model(label, expected_sha256, destination):
    print(f"Tek bir .pt seç: {label}")
    uploaded = files.upload()
    candidates = [(name, data) for name, data in uploaded.items() if name.lower().endswith(".pt")]
    if len(candidates) != 1:
        raise RuntimeError(f"{label} için tam olarak bir adet .pt seçmelisin")
    uploaded_name, uploaded_bytes = candidates[0]
    actual = hashlib.sha256(uploaded_bytes).hexdigest()
    if actual != expected_sha256:
        raise RuntimeError(f"{label} SHA-256 yanlış: {actual}; beklenen: {expected_sha256}")
    destination.write_bytes(uploaded_bytes)
    uploaded_path = Path(uploaded_name)
    if uploaded_path.resolve() != destination.resolve():
        uploaded_path.unlink(missing_ok=True)
    print(f"Yüklendi ve yeniden adlandırıldı: {uploaded_name} -> {destination.name}")

TASK1_MODEL_DIR.mkdir(parents=True, exist_ok=True)

DINOV2_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
if not DINOV2_MODEL_PATH.is_file() or sha256_path(DINOV2_MODEL_PATH) != DINOV2_SHA256:
    temporary = DINOV2_MODEL_PATH.with_suffix(".pth.part")
    temporary.unlink(missing_ok=True)
    request = Request(DINOV2_URL, headers={"User-Agent": "HYZ-2026-Colab"})
    try:
        with urlopen(request, timeout=240) as response, temporary.open("wb") as output:
            while chunk := response.read(1024 * 1024):
                output.write(chunk)
        actual = sha256_path(temporary)
        if temporary.stat().st_size != DINOV2_SIZE_BYTES or actual != DINOV2_SHA256:
            raise RuntimeError(f"DINOv2 artifact doğrulaması başarısız: size={temporary.stat().st_size} sha={actual}")
        temporary.replace(DINOV2_MODEL_PATH)
    finally:
        temporary.unlink(missing_ok=True)
print(f"DINOv2 hazır: {DINOV2_MODEL_PATH.name}")


### 2A. Object detection modelini yükle

Bu hücrede normal object detection `.pt` dosyanı seç. Dosya adı farklı olabilir; doğrulandıktan sonra otomatik olarak `yolo26n-1.1.0.pt` adına yazılır.


In [ ]:
upload_verified_model(
    "object detection modeli",
    DETECTOR_SHA256,
    DETECTOR_PATH,
)


### 2B. Landing classification modelini yükle

Bu hücrede classification `cls.pt` dosyanı seç. Dosya adı farklı olabilir; doğrulandıktan sonra otomatik olarak `landing_cls.pt` adına yazılır.


In [ ]:
upload_verified_model(
    "landing classification modeli",
    LANDING_CLASSIFIER_SHA256,
    LANDING_CLASSIFIER_PATH,
)


### 2C. Model dosyalarını doğrula


In [ ]:
for label, path, expected in (
    ("detector", DETECTOR_PATH, DETECTOR_SHA256),
    ("landing classifier", LANDING_CLASSIFIER_PATH, LANDING_CLASSIFIER_SHA256),
    ("DINOv2 ViT-S/14", DINOV2_MODEL_PATH, DINOV2_SHA256),
):
    if not path.is_file():
        raise FileNotFoundError(f"Task 1 {label} eksik: {path}")
    actual = sha256_path(path)
    if actual != expected:
        raise RuntimeError(f"Task 1 {label} doğrulaması başarısız: {actual}")
    print(f"Task 1 {label} hazır: {path.name} ({path.stat().st_size / 1024 / 1024:.1f} MiB) SHA={actual[:12]}")


## 3. Python 3.12 ve bağımlılıklar

Kaynakları pinlenen üç görev modülünün güncel bağımlılıkları çözülür. Colab runtime türü **T4 GPU** olmalıdır.


In [ ]:
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.12"], check=True)
subprocess.run(
    ["uv", "sync", "--python", "3.12"],
    cwd=REPOSITORY_DIR,
    check=True,
)
subprocess.run(
    [
        "uv", "pip", "install",
        "--python", str(REPOSITORY_DIR / ".venv/bin/python"),
        "torch==2.13.0", "torchvision==0.28.0",
    ],
    check=True,
)
print("Python 3.12 ve HYZ runtime bağımlılıkları hazır.")


## 4. Modül preflight

Bu hücre gerçek checkpoint'leri yükler ve gönderimden önce GPU, Task 1 model sınıfları, Task 2 config'i ve Task 3 production backend'ini doğrular. Herhangi bir hata varsa yarışmayı başlatma.


In [ ]:
preflight_code = r'''
import os
from pathlib import Path
import torch
from hyz_task1 import Task1Module
from hyz_task3 import build_runtime as build_task3_runtime
from team_solution.task2 import Task2Module

os.environ["MPLBACKEND"] = "Agg"
root = Path.cwd()
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU bulunamadı; Colab > Runtime > Change runtime type > T4 GPU seç")
print(f"GPU READY: {torch.cuda.get_device_name(0)} | torch={torch.__version__} | cuda={torch.version.cuda}", flush=True)
print("[1/3] Task 1 modelleri yükleniyor...", flush=True)
task1 = Task1Module(device="cuda")
print("[1/3] Task 1 OK", flush=True)
print("[2/3] Task 2 runtime yükleniyor...", flush=True)
task2 = Task2Module.from_config_path(
    root / "modules/task2_trajectory/config/team_solution.yaml",
    variant="adaptive",
)
print("[2/3] Task 2 OK", flush=True)
print("[3/3] Task 3 DINOv2 yükleniyor...", flush=True)
task3 = build_task3_runtime(root / "config/task3.yaml")
if not task3.enabled:
    raise RuntimeError("Task 3 production runtime disabled")
if task3.backend != "dinov2":
    raise RuntimeError(f"Task 3 backend dinov2 değil: {task3.backend}")
if str(task3.module._engine.search.device) != "cuda":
    raise RuntimeError(f"Task 3 DINOv2 CUDA üzerinde değil: {task3.module._engine.search.device}")
print("[3/3] Task 3 OK", flush=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Task 1: detector={task1.config.model_path.name} landing={task1.config.landing_model_path.name}")
if task2.config.runtime.mode != "production":
    raise RuntimeError(f"Task 2 runtime mode production değil: {task2.config.runtime.mode}")
print(f"Task 2: variant=adaptive mode={task2.config.runtime.mode}")
print(f"Task 3: enabled={task3.enabled} backend={task3.backend}")
'''
module_preflight = subprocess.run(
    [str(VENV_PYTHON), "-c", preflight_code],
    cwd=REPOSITORY_DIR,
    env={**os.environ, "MPLBACKEND": "Agg"},
    text=True,
    capture_output=True,
)
print(module_preflight.stdout, end="")
if module_preflight.returncode != 0:
    print(module_preflight.stderr, end="")
    raise RuntimeError(
        f"Modül preflight başarısız (exit={module_preflight.returncode}); gerçek traceback yukarıda."
    )
MODULE_PREFLIGHT_OK = True
print("MODULE_PREFLIGHT_OK=True")


## 5. Yarışma ayarları

Secret değerleri ignored `config/.env` dosyasına güvenli izinlerle yazılır. Değerler notebook çıktısında gösterilmez.


In [ ]:
def required_secret(name):
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Colab Secrets içinde {name} eksik")
    return str(value)

def quote_env(value):
    clean = value.replace("\n", "").replace("\r", "")
    return '"' + clean.replace("\\", "\\\\").replace('"', '\\"') + '"'

environment_values = {
    "TEAM_NAME": required_secret("TEAM_NAME"),
    "PASSWORD": required_secret("PASSWORD"),
    "EVALUATION_SERVER_URL": required_secret("EVALUATION_SERVER_URL").rstrip("/") + "/",
    "SESSION_NAME": required_secret("SESSION_NAME"),
}
environment_path = REPOSITORY_DIR / "config/.env"
environment_path.write_text(
    "".join(f"{key}={quote_env(value)}\n" for key, value in environment_values.items()),
    encoding="utf-8",
)
environment_path.chmod(0o600)
del environment_values
print("Private config/.env hazır; secret değerleri gösterilmedi.")


## 6. Sunucu preflight — prediction göndermez

Yalnız login ve `/progress/` kontrolü yapar. Frame istemez ve prediction göndermez. Aktif oturum yoksa yarışma başlatma hücresi güvenli biçimde bekler.


In [ ]:
server_preflight_code = r'''
from pathlib import Path
from decouple import Config, RepositoryEnv
from TAKIM_BAGLANTI_ARAYUZU.src.connection_handler import ConnectionHandler

env = Config(RepositoryEnv(str(Path.cwd() / "config/.env")))
print("[SERVER 1/3] Login deneniyor...", flush=True)
server = ConnectionHandler(
    env("EVALUATION_SERVER_URL").rstrip("/") + "/",
    username=env("TEAM_NAME"),
    password=env("PASSWORD"),
)
if not server.auth_token:
    raise RuntimeError("Yarışma sunucusu login başarısız")
print("[SERVER 1/3] Login OK", flush=True)
print("[SERVER 2/3] /progress/ okunuyor...", flush=True)
progress = server.get_progress()
if progress is None:
    raise RuntimeError("Yarışma sunucusu /progress/ okunamadı")
print(f"[SERVER 2/3] Progress OK: {progress}", flush=True)
print("[SERVER 3/3] SESSION_NAME doğrulanıyor...", flush=True)
expected_session = env("SESSION_NAME").strip()
actual_session = str(progress.get("session_name") or "").strip()
if not actual_session:
    raise RuntimeError("Sunucuda aktif session_name yok")
if actual_session != expected_session:
    raise RuntimeError(
        f"SESSION_NAME eşleşmiyor: Secret={expected_session!r}, server={actual_session!r}"
    )
print("[SERVER 3/3] SESSION_NAME OK", flush=True)
print(
    "Server hazır:",
    f"session={progress.get('session_name')}",
    f"frame={progress.get('frame_index')}/{progress.get('total_frames')}",
    f"completed={progress.get('completed')}",
)
'''
server_preflight = subprocess.run(
    [str(VENV_PYTHON), "-c", server_preflight_code],
    cwd=REPOSITORY_DIR,
    text=True,
    capture_output=True,
)
print(server_preflight.stdout, end="")
if server_preflight.returncode != 0:
    print(server_preflight.stderr, end="")
    raise RuntimeError(
        f"Server preflight başarısız (exit={server_preflight.returncode}); gerçek traceback yukarıda."
    )
SERVER_PREFLIGHT_OK = True
print("SERVER_PREFLIGHT_OK=True; prediction gönderilmedi.")


## 7. Yarışmayı başlat

Bütün üst hücreler hatasız bittikten ve aktif yarışma oturumu doğrulandıktan sonra `START_COMPETITION = True` yap. Bu hücre gerçek `/prediction/` gönderimlerini başlatır.

Task 1, Task 2 ve Task 3 aynı frame üzerinde paralel hazırlanır; state yalnız HTTP `201` veya idempotent `406` sonrasında birlikte commit edilir. Colab bağlantısı koparsa aynı runtime içinde hücreyi tekrar çalıştırmak `_logs` altındaki Task 2 snapshot'ını korur. Runtime tamamen sıfırlanmış ve server frame 0'dan ilerideyse eksik snapshot nedeniyle sistem fail-closed durur.


In [ ]:
START_COMPETITION = False

if START_COMPETITION:
    if not globals().get("MODULE_PREFLIGHT_OK"):
        raise RuntimeError("Modül preflight çalıştırılmadı veya başarısız")
    if not globals().get("SERVER_PREFLIGHT_OK"):
        raise RuntimeError("Sunucu preflight çalıştırılmadı veya başarısız")
    for path in (DETECTOR_PATH, LANDING_CLASSIFIER_PATH, DINOV2_MODEL_PATH, environment_path):
        if not path.is_file():
            raise RuntimeError(f"Başlatma için zorunlu dosya eksik: {path}")
    print("GERÇEK GÖNDERİM BAŞLIYOR: Task 1 + Task 2 + Task 3 aktif")
    process = subprocess.Popen(
        [str(VENV_PYTHON), "-u", "main.py"],
        cwd=REPOSITORY_DIR,
        env={**os.environ, "PYTHONUNBUFFERED": "1", "MPLBACKEND": "Agg"},
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
        exit_code = process.wait()
        if exit_code:
            raise RuntimeError(f"main.py exit code {exit_code}; gerçek hata yukarıda")
    except KeyboardInterrupt:
        process.terminate()
        process.wait(timeout=15)
        print("Runner kullanıcı tarafından durduruldu. Son kabul edilen frame snapshot'ı korundu.")
else:
    print("Güvenli bekleme: START_COMPETITION=False; prediction gönderilmedi.")


## Yarışma kontrol listesi

Başlatmadan hemen önce:

- Colab runtime: **T4 GPU**
- Üç kaynak pini hücre çıktısında doğru
- Detector, `cls.pt` ve DINOv2 SHA doğrulaması geçti
- `MODULE_PREFLIGHT_OK=True`
- `SERVER_PREFLIGHT_OK=True` ve doğru aktif session görünüyor
- Yalnız bundan sonra `START_COMPETITION=True`
